In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="py_mini_racer")
warnings.filterwarnings("ignore", message=".*pkg_resources is deprecated.*")
warnings.filterwarnings("ignore", module="baostock")
import os
from dotenv import load_dotenv
from utils import ROOT_DIR, fetch_baostock_data, query_zz500_stocks
import clickhouse_connect
import pandas as pd
import numpy as np
import torch
from script.alphanet import AlphaNet, get_trade_calender, process_raw_data
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime, timedelta

In [ ]:
load_dotenv()

client = clickhouse_connect.get_client(
    host=os.getenv('CH_HOST'),
    port=8123,
    username=os.getenv('CH_USERNAME'),
    password=os.getenv('CH_PASSWORD'),
    database=os.getenv('CH_DB'),
    compress=False,
    connect_timeout=60,
    send_receive_timeout=6000
)

In [ ]:
backtest_start_date, backtest_end_date = '2025-01-01', '2026-01-01'
zz500_code = query_zz500_stocks()['code']
zz500_code = [code[3:] for code in zz500_code]

code_str = ", ".join(f"'{code}'" for code in zz500_code)

raw_df = client.query_df(f"""
    SELECT * FROM level1.v_stock_daily_online
    WHERE trade_date BETWEEN '{backtest_start_date}' AND '{backtest_end_date}'
        AND stock_code IN ({code_str})
    ORDER BY trade_date, stock_code
""")

trade_calender = get_trade_calender(backtest_start_date, backtest_end_date)
X, y_raw = process_raw_data(raw_df, trade_calender, keep_time_series=True)

step = 5
X_trade = X[::step, ...]
y_trade = y_raw[::step, ...]

In [ ]:
COMMISSION_RATE = 0.002
n_groups = 5
stocks = sorted(raw_df['stock_code'].unique())
S = len(stocks)
T, S = y_trade.shape
group_returns = np.full((T, n_groups), np.nan)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AlphaNet(input_channels=8, op_d=10, hidden_dim=30, window_size=30).to(device)
model_path = ROOT_DIR / '1-weights' / 'best_alpha_net.pth'
model.load_state_dict(torch.load(model_path, map_location=device))
prev_holdings = [set() for _ in range(n_groups)]
for i, (X_cs, y_cs) in enumerate(zip(X_trade, y_trade)):
    cs_valid = ~np.any(np.isnan(X_cs), axis=(1, 2)) & ~np.isnan(y_cs)
    valid_indices = np.where(cs_valid)[0]
    if len(valid_indices) == 0:
        for g in range(n_groups):
            group_returns[i, g] = 0.0
        continue
    X_valid = X_cs[cs_valid]
    y_valid = y_cs[cs_valid]
    with torch.no_grad():
        scores = model(torch.tensor(X_valid, dtype=torch.float32, device=device)
                       ).squeeze().cpu().numpy()
    ranks = pd.Series(scores).rank(pct=True, method='first')
    groups = np.floor(ranks * n_groups).clip(0, n_groups - 1).astype(int)
    curr_holdings = [set() for _ in range(n_groups)]
    for idx, g in enumerate(groups):
        global_idx = valid_indices[idx]
        curr_holdings[g].add(global_idx)
    for g in range(n_groups):
        mask = (groups == g)
        if mask.sum() == 0:
            group_returns[i, g] = 0.0
            continue
        gross_ret = np.mean(y_valid[mask])
        prev_set = prev_holdings[g]
        curr_set = curr_holdings[g]
        if i == 0:
            cost_rate = COMMISSION_RATE
        else:
            n_new = len(curr_set)
            n_old = len(prev_set)
            if n_new == 0:
                cost_rate = 0.0
            else:
                overlap = len(curr_set & prev_set)
                turnover = 1.0 - overlap / n_new
                cost_rate = 2 * COMMISSION_RATE * turnover
        net_ret = gross_ret - cost_rate
        group_returns[i, g] = net_ret
    prev_holdings = curr_holdings
group_returns = np.nan_to_num(group_returns, nan=0.0)
cum_returns = np.cumprod(1 + group_returns, axis=0)

In [ ]:
trade_dates = pd.to_datetime(trade_calender['trade_date'].values)
window_size = 30
start_idx = window_size - 1
indices = np.arange(start_idx, start_idx + T * step, step)[:T]
rebalance_dates = trade_dates[indices]
initial_date = rebalance_dates[0] - pd.Timedelta(days=1)
plot_dates = pd.DatetimeIndex([initial_date] + list(rebalance_dates))
plot_values = np.vstack([np.ones(n_groups), cum_returns])
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(18, 7))
if n_groups <= 10:
    colors = plt.cm.tab10.colors
elif n_groups <= 20:
    colors = plt.cm.tab20.colors
else:
    colors = sns.color_palette("husl", n_groups)
for g in range(n_groups):
    ax.plot(plot_dates, plot_values[:, g],
            color=colors[g], linewidth=2.0, alpha=0.8,
            label=f'Group {g+1}')

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
ax.xaxis.set_minor_formatter(mdates.DateFormatter('%b'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=12)
plt.setp(ax.xaxis.get_minorticklabels(), fontsize=9)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))

ax.set_title(f'Cumulative Returns of AlphaNet Factor Groups '
             f'({n_groups} Groups, {step}d Rebalancing)',
             fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Date', fontsize=14)
ax.set_ylabel('Cumulative Return', fontsize=14)

if n_groups <= 10:
    legend_ncol = 1
    legend_fontsize = 12
    legend_loc = 'upper left'
    bbox_anchor = None
    right_margin = 0.9
else:
    legend_ncol = 2 if n_groups <= 20 else 3
    legend_fontsize = 10
    legend_loc = 'upper left'
    bbox_anchor = (1.02, 1.0)
    right_margin = 0.75
ax.legend(loc=legend_loc, bbox_to_anchor=bbox_anchor,
          frameon=True, fontsize=legend_fontsize, shadow=True,
          ncol=legend_ncol)
plt.subplots_adjust(right=right_margin)
ax.grid(True, linestyle='--', alpha=0.5)
ax.axhline(y=1.0, color='black', linestyle='--', linewidth=1, alpha=0.6)
plt.show()

In [ ]:
trade_end_date = datetime.today()
trade_start_date = trade_end_date - timedelta(days=2 * window_size)
trade_start_date, trade_end_date = trade_start_date.strftime("%Y-%m-%d"), trade_end_date.strftime("%Y-%m-%d")

_calender = get_trade_calender(trade_start_date, trade_end_date)
zz500_dict = fetch_baostock_data(zz500_code, trade_start_date, trade_end_date, persistent=True)

In [ ]:
column_mapping = {
    'Open': 'open',
    'Close': 'close',
    'High': 'high',
    'Low': 'low',
    'Volume': 'volume',
    'Amount': 'amount',
    'pctChg': 'pct_chg',
    'turn': 'turnover_rate'
}

feature_cols = ['open', 'close', 'high', 'low', 'volume', 'amount', 'pct_chg', 'turnover_rate']

raw_lst = []
for code, stock_df in zz500_dict.items():
    stock_df = stock_df.rename(columns=column_mapping)
    stock_df = stock_df[feature_cols]
    arr = stock_df[-30:].values.swapaxes(0, 1)
    raw_lst.append(arr[np.newaxis, :, :])

cs = np.concatenate(raw_lst, axis=0)
cs_ts = torch.tensor(cs, dtype=torch.float32).to(device)
pred = model(cs_ts).squeeze()

values, indices = torch.topk(pred, k=20)
top50_indices = indices.tolist()
zz500_code_np = np.array(zz500_code)
target_code = zz500_code_np[top50_indices]
target_code = [code for code in target_code if not code.startswith('688')]
print(target_code)
print(len(target_code))